# Lab 9 — Clustering with K-Means

**Course:** Data Mining Lab  
**Notebook:** Lab 9 — Clustering with K-Means

## Objectives
- Use distance-based K-Means clustering.
- Select k with an elbow plot, visualize clusters and interpret cluster profiles.

> Run cells from top to bottom. The notebook uses a reproducible random seed. For Adult Income labs, it attempts to load the public UCI Adult dataset and uses a clearly labelled local synthetic fallback if the download is unavailable.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
COLUMNS = ["age","workclass","fnlwgt","education","education.num","marital.status",
           "occupation","relationship","race","sex","capital.gain","capital.loss",
           "hours.per.week","native.country","income"]

def load_adult():
    local = Path("adult.csv")
    if local.exists():
        data = pd.read_csv(local)
        if set(COLUMNS).issubset(data.columns):
            return data[COLUMNS].copy(), "local adult.csv"
    try:
        data = pd.read_csv(DATA_URL, names=COLUMNS, na_values="?", skipinitialspace=True)
        return data, "UCI Adult dataset (downloaded)"
    except Exception as exc:
        rng = np.random.default_rng(42)
        n = 1200
        age = rng.integers(18, 75, n)
        hours = np.clip(rng.normal(40, 12, n).round(), 1, 80).astype(int)
        edu = rng.integers(1, 16, n)
        gain = rng.choice([0,0,0,0,500,1500,5000,10000], n)
        score = -5 + 0.055*age + 0.10*edu + 0.025*hours + (gain>0)*1.0
        income = np.where(rng.random(n) < 1/(1+np.exp(-score)), ">50K", "<=50K")
        data = pd.DataFrame({
            "age":age,"workclass":rng.choice(["Private","Self-emp","Government"],n),
            "fnlwgt":rng.integers(20000,500000,n),
            "education":rng.choice(["HS-grad","Bachelors","Masters","Some-college"],n),
            "education.num":edu,"marital.status":rng.choice(["Married","Never-married","Divorced"],n),
            "occupation":rng.choice(["Tech","Sales","Service","Admin"],n),
            "relationship":rng.choice(["Husband","Not-in-family","Own-child"],n),
            "race":rng.choice(["White","Black","Other"],n),"sex":rng.choice(["Male","Female"],n),
            "capital.gain":gain,"capital.loss":rng.choice([0,0,0,500,1000],n),
            "hours.per.week":hours,"native.country":rng.choice(["United-States","Other"],n),
            "income":income})
        print("NOTICE: Public dataset unavailable; using reproducible synthetic teaching data. Reason:", str(exc)[:180])
        return data, "synthetic fallback (not UCI observations)"

df, DATA_SOURCE = load_adult()
print("Data source:", DATA_SOURCE, "| Shape:", df.shape)
display(df.head())

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
features = df[["age","education.num","capital.gain","capital.loss","hours.per.week"]].copy()
features = features.fillna(features.median())
scaled = StandardScaler().fit_transform(features)

## Elbow method

In [ ]:
inertias=[]
for k in range(2,9):
    km=KMeans(n_clusters=k,random_state=42,n_init=10).fit(scaled)
    inertias.append(km.inertia_)
plt.plot(range(2,9),inertias,marker="o")
plt.xlabel("Number of clusters (k)"); plt.ylabel("Inertia"); plt.title("Elbow method"); plt.show()

## Fit K-Means and visualize clusters

In [ ]:
kmeans=KMeans(n_clusters=3,random_state=42,n_init=10)
labels=kmeans.fit_predict(scaled)
print("Silhouette score:", round(silhouette_score(scaled,labels),4))
pca=PCA(n_components=2,random_state=42)
coords=pca.fit_transform(scaled)
plt.figure(figsize=(7,5))
plt.scatter(coords[:,0],coords[:,1],c=labels,s=12,alpha=.6)
plt.title("K-Means clusters (PCA projection)"); plt.xlabel("PC1"); plt.ylabel("PC2"); plt.show()
clustered=features.assign(cluster=labels)
display(clustered.groupby("cluster").mean().round(2))